# SenseVoice 数据量消融实验

验证不同数据量（25%/50%/75%）和两种数据增强策略对微调效果的影响。

In [ ]:
def add_real_noise(audio_path, noise_path, snr_db_range=(10, 20)):
    """叠加真实噪音"""
    wav, sr = torchaudio.load(audio_path)
    noise, noise_sr = torchaudio.load(noise_path)
    if noise_sr != sr:
        noise = torchaudio.functional.resample(noise, noise_sr, sr)
    if noise.shape[1] < wav.shape[1]:
        pad = torch.zeros(wav.shape[1] - noise.shape[1])
        noise = torch.cat([noise, pad.unsqueeze(0)], dim=1)
    else:
        noise = noise[:, :wav.shape[1]]
    snr_db = random.uniform(*snr_db_range)
    signal_power = (wav ** 2).mean()
    noise_power = (noise ** 2).mean()
    if noise_power < 1e-10:
        noise_power = 1e-10
    adjusted_noise = noise * torch.sqrt(signal_power / (noise_power * (10 ** (snr_db / 10))))
    noisy_wav = wav + adjusted_noise
    return noisy_wav, sr